## Example 1: A task's lifecycle in FastA2A's in-memory storage

In [ ]:
import fasta2a
from fasta2a.storage import InMemoryStorage

# Example 1: a task's lifecycle in FastA2A's in-memory storage
# (fasta2a is pydantic's Agent-to-Agent protocol framework - it has nothing to do with FASTA sequence files)
print("fasta2a - Task lifecycle in InMemoryStorage")
assert {"FastA2A", "Skill", "Storage", "Broker", "Worker"} <= set(fasta2a.__all__), fasta2a.__all__


async def task_lifecycle():
    storage = InMemoryStorage()
    request = {"role": "user", "parts": [{"kind": "text", "text": "hello"}], "kind": "message", "message_id": "m1"}
    task = await storage.submit_task("ctx-1", request)
    assert task["status"]["state"] == "submitted" and task["context_id"] == "ctx-1", task
    assert task["history"][0]["task_id"] == task["id"], task["history"]  # the message is bound to its task
    print(f"✓ Submitted task {task['id']} in context {task['context_id']}")

    working = await storage.update_task(task["id"], state="working")
    assert working["status"]["state"] == "working", working

    reply = {"role": "agent", "parts": [{"kind": "text", "text": "done"}], "kind": "message", "message_id": "m2"}
    artifact = {"artifact_id": "a1", "parts": [{"kind": "text", "text": "HELLO"}]}
    done = await storage.update_task(task["id"], state="completed", new_artifacts=[artifact], new_messages=[reply])
    loaded = await storage.load_task(task["id"])
    assert loaded == done and loaded["status"]["state"] == "completed", loaded
    assert [m["message_id"] for m in loaded["history"]] == ["m1", "m2"] and loaded["artifacts"] == [artifact], loaded
    print(f"✓ Task completed with {len(loaded['history'])} messages and {len(loaded['artifacts'])} artifact")

    # history_length keeps only the most recent messages; an unknown task id loads as None (measured)
    assert [m["message_id"] for m in (await storage.load_task(task["id"], history_length=1))["history"]] == ["m2"]
    assert await storage.load_task("no-such-task") is None
    return loaded


task = await task_lifecycle()
print(f"✓ Final state: {task['status']['state']}")

print("Example 1 completed")

## Example 2: The agent card lists the agent's skills

In [ ]:
# Example 2: The agent card lists the agent's skills
import asyncio, json
from fasta2a import FastA2A, Skill
from fasta2a.broker import InMemoryBroker
from fasta2a.storage import InMemoryStorage


async def asgi_request(app, method, path, payload=None):
    """Send one HTTP request to an ASGI app in this process; return (status, response body)."""
    body = b"" if payload is None else json.dumps(payload).encode()
    incoming = [{"type": "http.request", "body": body, "more_body": False}]
    outgoing = []

    async def receive():
        if incoming:
            return incoming.pop(0)
        await asyncio.Event().wait()  # the client stays connected until the response is complete

    async def send(message):
        outgoing.append(message)

    scope = {"type": "http", "asgi": {"version": "3.0"}, "http_version": "1.1", "method": method, "scheme": "http",
             "path": path, "raw_path": path.encode(), "root_path": "", "query_string": b"",
             "headers": [(b"host", b"agent.test"), (b"content-type", b"application/json")],
             "client": ("127.0.0.1", 50000), "server": ("agent.test", 80)}
    await app(scope, receive, send)
    status = next(m["status"] for m in outgoing if m["type"] == "http.response.start")
    return status, b"".join(m.get("body", b"") for m in outgoing if m["type"] == "http.response.body")


skills = [
    Skill(id="shout", name="Shout", description="Repeats the text in upper case", tags=["text"],
          input_modes=["text/plain"], output_modes=["text/plain"]),
    Skill(id="count", name="Count", description="Counts the words", tags=["text", "stats"],
          input_modes=["text/plain"], output_modes=["application/json"]),
]
app = FastA2A(storage=InMemoryStorage(), broker=InMemoryBroker(), name="Notebook agent", version="0.1.0",
              description="Answers in upper case", skills=skills)


async def fetch_card():
    async with app.task_manager:  # what the app's own lifespan does when a server starts it
        return await asgi_request(app, "GET", "/.well-known/agent-card.json")

status, body = await fetch_card()
card = json.loads(body)
assert status == 200 and card["name"] == "Notebook agent" and card["version"] == "0.1.0", (status, card)
assert [skill["id"] for skill in card["skills"]] == ["shout", "count"], card["skills"]
# The card uses the protocol's camelCase field names.
assert card["skills"][1]["tags"] == ["text", "stats"] and card["skills"][1]["outputModes"] == ["application/json"]
assert card["capabilities"]["streaming"] is True, card["capabilities"]
print(f"OK -- the agent card names {len(card['skills'])} skills:", [skill["name"] for skill in card["skills"]])

## Example 3: A JSON-RPC message/send creates a task, and a worker completes it

In [ ]:
# Example 3: A JSON-RPC message/send creates a task, and a worker completes it
import asyncio, json, uuid
from fasta2a import FastA2A, Worker
from fasta2a.broker import InMemoryBroker
from fasta2a.storage import InMemoryStorage


class ShoutWorker(Worker):
    """Answers each task with its text in upper case; the text 'fail' makes it raise."""

    async def run_task(self, params):
        task = await self.storage.load_task(params["id"])
        await self.storage.update_task(task["id"], state="working")
        text = " ".join(part.get("text", "") for message in task["history"] for part in message["parts"])
        if text == "fail":
            raise ValueError("asked to fail")
        artifact = {"artifact_id": str(uuid.uuid4()), "name": "shouted", "parts": [{"text": text.upper()}]}
        reply = {"role": "agent", "parts": [{"text": "done"}], "message_id": str(uuid.uuid4())}
        await self.storage.update_task(task["id"], state="completed", new_artifacts=[artifact], new_messages=[reply])

    async def cancel_task(self, params):
        await self.storage.update_task(params["id"], state="canceled")

    def build_message_history(self, history):
        return history

    def build_artifacts(self, result):
        return []


def rpc(method, params, request_id=1):
    return {"jsonrpc": "2.0", "id": request_id, "method": method, "params": params}


def new_agent():
    storage, broker = InMemoryStorage(), InMemoryBroker()
    return FastA2A(storage=storage, broker=broker), ShoutWorker(broker=broker, storage=storage)


async def wait_until_done(app, task_id):
    for _ in range(500):
        status, body = await asgi_request(app, "POST", "/", rpc("tasks/get", {"id": task_id}, 2))
        task = json.loads(body)["result"]
        if task["status"]["state"] in ("completed", "failed", "canceled"):
            return task
        await asyncio.sleep(0.01)
    raise TimeoutError(f"task {task_id} did not finish")


async def send_and_wait():
    app, worker = new_agent()
    async with app.task_manager, worker.run():
        message = {"role": "user", "parts": [{"text": "hello agent"}], "messageId": "m-1", "contextId": "ctx-1"}
        status, body = await asgi_request(app, "POST", "/", rpc("message/send", {"message": message}))
        sent = json.loads(body)
        assert status == 200 and sent["id"] == 1 and "error" not in sent, sent
        created = sent["result"]["task"]
        assert created["contextId"] == "ctx-1" and created["history"][0]["messageId"] == "m-1", created
        return await wait_until_done(app, created["id"])

task = await send_and_wait()
assert task["status"]["state"] == "completed", task["status"]
assert [message["role"] for message in task["history"]] == ["user", "agent"], task["history"]
assert task["artifacts"][0]["name"] == "shouted" and task["artifacts"][0]["parts"] == [{"text": "HELLO AGENT"}], task
print(f"OK -- task {task['id']} completed in context {task['contextId']}:", task["artifacts"][0]["parts"][0]["text"])

## Example 4: An unknown task id is a JSON-RPC error, and a worker that raises leaves its task failed

In [ ]:
# Example 4: An unknown task id is a JSON-RPC error, and a worker that raises leaves its task failed
import json


async def errors():
    app, worker = new_agent()
    async with app.task_manager, worker.run():
        status, body = await asgi_request(app, "POST", "/", rpc("tasks/get", {"id": "no-such-task"}, 3))
        missing = json.loads(body)
        message = {"role": "user", "parts": [{"text": "fail"}], "messageId": "m-2"}
        status, body = await asgi_request(app, "POST", "/", rpc("message/send", {"message": message}, 4))
        failed = await wait_until_done(app, json.loads(body)["result"]["task"]["id"])
    return missing, failed

missing, failed = await errors()
assert missing["id"] == 3 and "result" not in missing, missing
assert missing["error"] == {"code": -32001, "message": "Task not found"}, missing["error"]
assert failed["status"]["state"] == "failed" and "artifacts" not in failed, failed
print("OK -- unknown task:", missing["error"]["message"], "| the raising worker left its task", failed["status"]["state"])

## Example 5: A message/stream request sends the new task, then its status updates, as server-sent events

In [ ]:
# Example 5: A message/stream request sends the new task, then its status updates, as server-sent events
import json


async def stream():
    app, worker = new_agent()
    async with app.task_manager, worker.run():
        message = {"role": "user", "parts": [{"text": "stream me"}], "messageId": "m-3"}
        status, body = await asgi_request(app, "POST", "/", rpc("message/stream", {"message": message}, 5))
        assert status == 200, status
        events = [json.loads(chunk[len("data: "):]) for chunk in body.decode().split("\n\n") if chunk.startswith("data: ")]
        final = await wait_until_done(app, events[0]["result"]["task"]["id"])
    return events, final

events, final = await stream()
assert len(events) >= 2 and all(event["id"] == 5 for event in events), events
first, last = events[0]["result"], events[-1]["result"]
assert first["task"]["history"][0]["messageId"] == "m-3", first  # the first event carries the new task
assert last["statusUpdate"]["taskId"] == first["task"]["id"], last
assert last["statusUpdate"]["status"]["state"] == "completed", last  # the last one says the task is over
assert final["artifacts"][0]["parts"] == [{"text": "STREAM ME"}], final
print(f"OK -- {len(events)} events streamed; the task ended", last["statusUpdate"]["status"]["state"])